# OCR v2 — 4-worker recognition (canary first)

Chỉ chạy model trên Kaggle T4. Internet ON, Kaggle Secret **HF_TOKEN** có quyền ghi repo private.
Không gọi API, không chạy lại EasyOCR/Vintern, không làm nét. Notebook xuất recognition/selection
evidence, **chưa xuất terminal envelope hoặc SQLite Online**. Xem runbook OCR v2 production.

1. Gắn dataset Kaggle chứa **frames.csv + frames.csv.state.json**. Một notebook **CPU**
   chạy `ACTION='plan'` một lần: đọc catalog từ Kaggle Input, tải/validate 9 archive OCR từ HF,
   không đọc JPEG và không chạy OCR. Tải `ocr-v2-worker-plan.json` về rồi gắn cùng file này
   vào bốn notebook worker (cùng input revision, không tạo lại bốn plan).
2. Trên mỗi tài khoản gắn cùng catalog/state và JPEG datasets đúng batch do plan phân công, chọn T4,
   `ACTION='run'`, `WORKER_SLOT=1..4`, `RUN_MODE='canary'`. Mặc định dừng sau 1 minibatch
   đã verify HF. Đặt `INTERRUPT_AFTER_MINIBATCHES=0` rồi chạy lại trong process/session mới.
3. Canary hoàn tất sẽ in `report_sha256`. Review kết quả/elapsed và copy hash vào
   `APPROVED_CANARY_SHA256` trên đúng worker trước khi chọn `RUN_MODE='production'`.
   Production không được coi ready; còn bước migration/union/SQLite và publishing gate.

Catalog/keyframes nằm trong dataset Kaggle; HF lưu archive/checkpoint/kết quả OCR và embedding.
`CATALOG_PATH=''` tự tìm dưới `INPUT_ROOT`; nhiều bản khác nhau thì điền exact path Kaggle.
Mất VM: chạy lại cùng notebook/plan/worker, gắn đúng catalog/state/keyframes. Chỉ checkpoint HF đã verify
là bền vững; phần sau checkpoint cuối có thể chạy lại. Không chạy trùng WORKER_SLOT đồng thời.
Checkpoint local mỗi minibatch, HF delta tối đa 5 phút giữa mốc kiểm tra + cuối pha/dừng chủ động.


In [ ]:
ACTION = 'plan'  # 'plan' chỉ một lần trên CPU; 'run' trên từng T4
HF_REPO_ID = 'MinhThuw0103/lastdance-visual-embeddings'
INPUT_REVISION = ''  # plan resolve một commit rồi khóa; worker dùng revision trong plan
INPUT_ROOT = '/kaggle/input'  # thư mục dataset được gắn vào notebook
CATALOG_PATH = ''  # tự tìm frames.csv trong INPUT_ROOT; hoặc điền exact path Kaggle (có state bên cạnh)
WORKER_PLAN = ''  # run: tự tìm ocr-v2-worker-plan.json duy nhất dưới /kaggle/input
KEYFRAMES_ROOT = INPUT_ROOT
WORKER_SLOT = 1  # bốn tài khoản dùng 1, 2, 3, 4 không trùng nhau
RUN_MODE = 'canary'
INTERRUPT_AFTER_MINIBATCHES = 1  # sau intentional stop, đổi về 0 rồi chạy lại
APPROVED_CANARY_SHA256 = ''  # bắt buộc trước production; hash report của đúng worker
RUN_SETUP = True  # run only; môi trường Gate B đúng pin có thể bỏ setup


In [ ]:
# Embedded source files; Gate B inference loop is not included.
SOURCES = {'kaggle_ocr_v2_production_runtime.py': '"""OCR v2 recognition worker. Import-safe: no GPU, network or filesystem side effects.\n\nOutputs are versioned recognition/selection evidence, NOT legacy terminal envelopes\nor an Online SQLite snapshot. Models run in separate Kaggle subprocesses.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport ast\nimport csv\nimport hashlib\nimport importlib.metadata as md\nimport io\nimport json\nimport math\nimport os\nimport re\nimport sqlite3\nimport subprocess\nimport sys\nimport threading\nimport time\nimport types\nimport unicodedata\nimport zipfile\nfrom collections import Counter\nfrom contextlib import contextmanager\nfrom pathlib import Path, PurePosixPath\n\nCONTRACT = "ocr-v2-recognition-worker-v1"\nBATCH_IDS = tuple(f"batch-{i:02d}" for i in range(1, 10))\nMODEL_NAMES = ("vietocr", "paddle")\nPOLICY = {"vietocr_low": 0.60, "paddle_override": 0.90, "decode_limit": 128,\n          "vietocr_batch": 64, "paddle_batch": 128, "crop": "pil_quad_v1_pad08_edge"}\nREFERENCE_NAMES = {\n    "VIETOCR_WEIGHT", "VIETOCR_CONFIGS", "PADDLE_MODEL", "PADDLE_MODEL_ID",\n    "CROP_SPEC_ID", "_float_list", "_edge_pad", "rectify_region_crop",\n    "sha256_file", "download_verified", "prepare_vietocr_model", "prepare_paddle_model",\n}\nREQUIRED = {"easyocr-frames.jsonl", "vintern-candidates.jsonl", "run-signature.json",\n            "batch-manifest.json", "SHA256SUMS"}\n\n\ndef encoded(value):\n    return json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":"),\n                      allow_nan=False).encode("utf-8")\n\n\ndef sha(payload):\n    return hashlib.sha256(payload).hexdigest()\n\n\ndef file_sha(path):\n    value = hashlib.sha256()\n    with Path(path).open("rb") as handle:\n        for block in iter(lambda: handle.read(8 * 1024 * 1024), b""):\n            value.update(block)\n    return value.hexdigest()\n\n\ndef atomic(path, payload):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_name(path.name + ".tmp")\n    with temporary.open("wb") as handle:\n        handle.write(payload)\n        handle.flush()\n        os.fsync(handle.fileno())\n    os.replace(temporary, path)\n\n\ndef log(event, **values):\n    print(event, encoded(values).decode("utf-8"), flush=True)\n\n\n@contextmanager\ndef heartbeat(stage, **identity):\n    stop, started = threading.Event(), time.monotonic()\n    def tick():\n        while not stop.wait(30):\n            log("HEARTBEAT", phase=stage, elapsed=round(time.monotonic() - started), **identity)\n    thread = threading.Thread(target=tick, daemon=True)\n    thread.start()\n    try:\n        yield\n    finally:\n        stop.set()\n        thread.join(timeout=1)\n\n\ndef reference_source(source):\n    nodes = []\n    for node in ast.parse(source).body:\n        name = node.name if isinstance(node, ast.FunctionDef) else None\n        if isinstance(node, ast.Assign) and isinstance(node.targets[0], ast.Name):\n            name = node.targets[0].id\n        if name in REFERENCE_NAMES:\n            nodes.append(node)\n    if len(nodes) != len(REFERENCE_NAMES):\n        raise ValueError("Gate B crop/model helper contract changed")\n    return ast.unparse(ast.Module(body=nodes, type_ignores=[])) + "\\n"\n\n\ndef helpers(model_dir):\n    source = reference_source(Path(__file__).with_name("kaggle_ocr_v2_gate_b_runtime.py").read_text(encoding="utf-8"))\n    namespace = {}\n    exec("from __future__ import annotations\\nimport hashlib, math, re, json, tarfile, urllib.request\\n"\n         "from pathlib import Path\\nfrom PIL import Image\\nfrom typing import Any\\n" + source, namespace)\n    Path(model_dir).mkdir(parents=True, exist_ok=True)\n    namespace["OUTPUT_ROOT"] = Path(model_dir)\n    return types.SimpleNamespace(**namespace), sha(source.encode())\n\n\ndef uid_for(video, shot, local):\n    return int.from_bytes(hashlib.blake2b(f"{video}:{shot}:{local}".encode(), digest_size=8).digest(), "big") >> 1\n\n\ndef uid_hash(values):\n    return sha("".join(f"{v}\\n" for v in sorted(values)).encode())\n\n\ndef catalog_hashes(path):\n    path = Path(path)\n    return {"catalog_sha256": file_sha(path),\n            "catalog_state_sha256": file_sha(path.with_name(path.name + ".state.json"))}\n\n\ndef resolve_catalog(config):\n    """Find the attached Kaggle catalog; never look for it in the HF output repo."""\n    explicit = config.get("catalog_path")\n    if explicit:\n        paths = [Path(explicit)]\n        if not paths[0].is_file():\n            raise ValueError(f"CATALOG_PATH does not exist: {explicit!r}; use a Kaggle Input file path")\n    else:\n        root = Path(config.get("input_root") or os.environ.get("AIC_DATA", "data"))\n        paths = sorted(p for p in root.rglob("frames.csv") if p.is_file())\n        if not paths:\n            raise ValueError(f"No frames.csv under {root}; attach the catalog Kaggle dataset "\n                             "or set CATALOG_PATH to its local file path. HF stores OCR artifacts.")\n    candidates = [{"path": str(p), "state": p.with_name(p.name + ".state.json").is_file()} for p in paths]\n    log("CATALOG_CANDIDATES", candidates=candidates)\n    if any(not entry["state"] for entry in candidates):\n        raise ValueError("Missing frames.csv.state.json beside a catalog candidate; attach the original "\n                         f"state or set CATALOG_PATH to a complete catalog pair: {candidates!r}")\n    # Batch datasets may carry identical full-catalog copies. Accept only byte-identical pairs.\n    identities = {tuple(catalog_hashes(p).values()) for p in paths}\n    if len(identities) != 1:\n        raise ValueError(f"Multiple different local catalogs; set CATALOG_PATH explicitly: {candidates!r}")\n    log("CATALOG_SELECTED", path=str(paths[0]), identical_copies=len(paths))\n    return paths[0]\n\n\ndef load_catalog(path):\n    path = Path(path)\n    state = json.loads(path.with_name(path.name + ".state.json").read_bytes())\n    if state.get("schema_version") != 1 or state.get("complete") is not True or state.get("csv_sha256") != file_sha(path):\n        raise ValueError("Catalog state/hash is not complete and valid")\n    result = {}\n    with path.open(encoding="utf-8", newline="") as handle:\n        reader = csv.DictReader(handle)\n        if reader.fieldnames != ["video_id", "local_idx", "frame_id", "pts_time", "shot_id", "window_id", "keyframe_uid"]:\n            raise ValueError("Unexpected frames.csv schema")\n        for row in reader:\n            for key in ("local_idx", "frame_id", "keyframe_uid"):\n                row[key] = int(row[key])\n            row["pts_time"] = float(row["pts_time"])\n            uid = row["keyframe_uid"]\n            if (uid in result or uid <= 0 or row["local_idx"] < 0 or row["frame_id"] < 0\n                    or not math.isfinite(row["pts_time"]) or row["pts_time"] < 0\n                    or uid != uid_for(row["video_id"], row["shot_id"], row["local_idx"])):\n                raise ValueError("Invalid catalog identity/mapping")\n            result[uid] = row\n    videos = {r["video_id"] for r in result.values()}\n    source_videos = [r["video_id"] for r in state.get("sources", [])]\n    if (len(result) != state.get("record_count") or len(videos) != state.get("video_count")\n            or len(source_videos) != len(set(source_videos)) or set(source_videos) != videos):\n        raise ValueError("Catalog counts/source videos mismatch")\n    return result\n\n\ndef load_archive(path, catalog):\n    """Validate archive byte checksums, canonical UID mapping and every region ID."""\n    with zipfile.ZipFile(path) as archive:\n        names = archive.namelist()\n        if (len(names) != len(set(names)) or not REQUIRED <= set(names)\n                or set(names) - REQUIRED - {"errors-history.jsonl"}):\n            raise ValueError("Unexpected/duplicate archive members")\n        checks = {}\n        for line in archive.read("SHA256SUMS").decode("ascii").splitlines():\n            digest, name = line.split(maxsplit=1)\n            if name in checks or not re.fullmatch(r"[0-9a-f]{64}", digest):\n                raise ValueError("Invalid archive SHA256SUMS")\n            checks[name] = digest\n        if set(checks) != set(names) - {"SHA256SUMS"}:\n            raise ValueError("Archive checksum coverage mismatch")\n        for name, digest in checks.items():\n            h = hashlib.sha256()\n            with archive.open(name) as handle:\n                for block in iter(lambda: handle.read(8 * 1024 * 1024), b""):\n                    h.update(block)\n            if h.hexdigest() != digest:\n                raise ValueError(f"Archive checksum mismatch: {name}")\n        manifest = json.loads(archive.read("batch-manifest.json"))\n        batch = manifest.get("batch_id")\n        if batch not in BATCH_IDS or manifest.get("tier") != "easyocr" or manifest.get("complete") is not True:\n            raise ValueError("Incomplete or wrong archive layer")\n        rows, seen, regions = [], set(), set()\n        with archive.open("easyocr-frames.jsonl") as handle:\n            for line in io.TextIOWrapper(handle, encoding="utf-8"):\n                if not line.strip():\n                    continue\n                row = json.loads(line)\n                uid = int(row["keyframe_uid"])\n                canonical = catalog.get(uid)\n                if uid in seen or canonical is None or any(row[k] != canonical[k] for k in ("video_id", "shot_id", "local_idx")):\n                    raise ValueError("Duplicate/foreign UID or catalog mapping drift")\n                seen.add(uid)\n                source = PurePosixPath(row["source_image"])\n                if source.is_absolute() or ".." in source.parts or "\\\\" in row["source_image"] or ":" in row["source_image"] or source.parent.name != row["video_id"]:\n                    raise ValueError("Invalid archive source_image")\n                row["frame_id"], row["pts_time"] = canonical["frame_id"], canonical["pts_time"]\n                if row.get("status") not in {"no_text", "text_detected", "error"}:\n                    raise ValueError("Unknown CRAFT frame status")\n                if row["status"] == "no_text" and row.get("regions"):\n                    raise ValueError("No-text frame contains regions")\n                for i, region in enumerate(row.get("regions") or []):\n                    box = region["bbox_px"]\n                    if len(box) != 8 or not all(math.isfinite(float(v)) for v in box):\n                        raise ValueError("Invalid source bbox")\n                    expected = sha(f"{uid}|{i}|{\';\'.join(f\'{v:.2f}\' for v in box)}".encode())[:24]\n                    rid = region["region_id"]\n                    if rid != expected or rid in regions:\n                        raise ValueError("Duplicate or unstable region ID")\n                    regions.add(rid)\n                rows.append(row)\n                if len(rows) % 10000 == 0:\n                    log("ARCHIVE_SCAN", batch=batch, frames=len(rows), regions=len(regions))\n    if (len(rows) != manifest.get("frames") or len(rows) != manifest.get("expected_frames")\n            or len(regions) != manifest.get("regions")\n            or uid_hash(seen) != manifest.get("assigned_uid_sha256")\n            or uid_hash(seen) != manifest.get("observed_uid_sha256")\n            or dict(Counter(r["status"] for r in rows)) != manifest.get("status")):\n        raise ValueError("Archive manifest/actual coverage mismatch")\n    return manifest, rows\n\n\ndef allocate(batches):\n    if set(batches) != set(BATCH_IDS):\n        raise ValueError("Need exactly nine batches")\n    loads, assignments = {str(i): 0 for i in range(1, 5)}, {str(i): [] for i in range(1, 5)}\n    for batch in sorted(batches, key=lambda b: (-batches[b]["regions"], b)):\n        worker = min(loads, key=lambda w: (loads[w], int(w)))\n        assignments[worker].append(batch)\n        loads[worker] += batches[batch]["regions"]\n    return assignments, loads\n\n\nclass HF:\n    """Read pinned inputs; write content-addressed evidence only to a private dataset."""\n    def __init__(self, repo):\n        from huggingface_hub import HfApi\n        self.token = os.environ.get("HF_TOKEN")\n        if not self.token:\n            try:\n                from kaggle_secrets import UserSecretsClient\n                self.token = UserSecretsClient().get_secret("HF_TOKEN")\n            except Exception:\n                pass\n        if not self.token:\n            raise RuntimeError("Add Kaggle Secret HF_TOKEN (write) before starting")\n        self.repo, self.api = repo, HfApi(token=self.token)\n        if not self.api.repo_info(repo_id=repo, repo_type="dataset").private:\n            raise RuntimeError("Expected a private OCR dataset")\n\n    def revision(self):\n        return str(self.api.repo_info(repo_id=self.repo, repo_type="dataset").sha)\n\n    def files(self, revision):\n        return self.api.list_repo_files(repo_id=self.repo, repo_type="dataset", revision=revision)\n\n    def download(self, name, revision):\n        from huggingface_hub import hf_hub_download\n        return Path(hf_hub_download(repo_id=self.repo, repo_type="dataset", filename=name,\n                                    revision=revision, token=self.token))\n\n    def put(self, path, name):\n        if not re.fullmatch(r"ocr/archives/batch-0[1-9]/ocr-v2/[0-9a-f]{64}/(?:canary|production)/.+", name):\n            raise ValueError("Refusing write outside v2 evidence namespace")\n        digest = file_sha(path)\n        for attempt in range(3):\n            try:\n                revision = self.revision()\n                if name not in self.files(revision):\n                    commit = self.api.upload_file(repo_id=self.repo, repo_type="dataset",\n                        path_or_fileobj=str(path), path_in_repo=name,\n                        commit_message=f"{name.split(\'/\')[2]} OCR v2 checkpoint")\n                    revision = commit.oid\n                downloaded = self.download(name, revision)\n                if file_sha(downloaded) != digest:\n                    raise ValueError("HF round-trip checksum mismatch")\n                log("HF_VERIFIED", artifact=name, sha256=digest, revision=revision)\n                return revision\n            except Exception as exc:\n                status = getattr(getattr(exc, "response", None), "status_code", None)\n                if status in {401, 403} or attempt == 2:\n                    raise RuntimeError("HF sync failed; local results kept; STOPPING inference") from None\n                log("HF_RETRY", attempt=attempt + 1, error=type(exc).__name__)\n                time.sleep(2 ** attempt)\n\n\ndef create_plan(config, hf):\n    catalog_path = resolve_catalog(config)\n    catalog = load_catalog(catalog_path)\n    identity = catalog_hashes(catalog_path)\n    revision = config.get("input_revision") or hf.revision()\n    files = hf.files(revision)\n    batches, all_uids, all_videos = {}, set(), set()\n    for batch in BATCH_IDS:\n        pattern = f"ocr-production-{batch}-easyocr.zip"\n        candidates = [n for n in files if n.startswith("ocr/archives/") and PurePosixPath(n).name == pattern]\n        if len(candidates) != 1:\n            raise ValueError(f"Expected one HF archive for {batch}")\n        with heartbeat("PLAN_ARCHIVE", batch=batch):\n            path = hf.download(candidates[0], revision)\n            manifest, rows = load_archive(path, catalog)\n        if manifest["batch_id"] != batch or manifest["catalog_sha256"] != identity["catalog_sha256"]:\n            raise ValueError("Archive batch/catalog hash drift")\n        uids = {r["keyframe_uid"] for r in rows}\n        videos = {r["video_id"] for r in rows}\n        if uids & all_uids or videos & all_videos:\n            raise ValueError("Batch partitions overlap")\n        all_uids.update(uids)\n        all_videos.update(videos)\n        batches[batch] = {"archive": candidates[0], "sha256": file_sha(path),\n                          "regions": manifest["regions"], "frames": len(rows),\n                          "uid_sha256": uid_hash(uids), "video_ids": sorted(videos)}\n        log("PLAN_BATCH_VALIDATED", batch=batch, **{k: batches[batch][k] for k in ("frames", "regions")})\n        del rows\n    if all_uids != set(catalog):\n        raise ValueError("Nine archives do not cover exactly frames.csv")\n    assignments, loads = allocate(batches)\n    plan = {"contract": CONTRACT, "repo": hf.repo, "input_revision": revision,\n            "catalog_source": "kaggle_input", "catalog": catalog_path.name, **identity,\n            "batches": batches, "assignments": assignments, "worker_regions": loads}\n    plan["plan_sha256"] = sha(encoded(plan))\n    atomic(Path(config["output"]) / "ocr-v2-worker-plan.json", encoded(plan))\n    log("PLAN_COMPLETE", assignments=assignments, regions=loads, plan_sha256=plan["plan_sha256"])\n    return plan\n\n\ndef validate_plan(plan):\n    body = {k: v for k, v in plan.items() if k != "plan_sha256"}\n    if plan.get("contract") != CONTRACT or sha(encoded(body)) != plan.get("plan_sha256"):\n        raise ValueError("Worker plan signature mismatch")\n    if plan.get("catalog_source") != "kaggle_input":\n        raise ValueError("Worker plan must use the attached Kaggle catalog; rerun ACTION=\'plan\' with this notebook")\n    assignments, loads = allocate(plan["batches"])\n    if plan["assignments"] != assignments or plan["worker_regions"] != loads:\n        raise ValueError("Worker assignment drift")\n    if not re.fullmatch(r"[0-9a-f]{40}", plan["input_revision"]):\n        raise ValueError("Input revision must be an immutable commit")\n\n\ndef normalized(text):\n    return " ".join(unicodedata.normalize("NFC", text).casefold().split())\n\n\ndef score(value):\n    try:\n        value = float(value)\n    except (TypeError, ValueError):\n        return None\n    return value if math.isfinite(value) and 0 <= value <= 1 else None\n\n\ndef guards(pred):\n    if pred is None:\n        return ["missing_prediction"]\n    reasons = []\n    text = pred["text"].strip()\n    if not text:\n        reasons.append("empty")\n    confidence = score(pred.get("confidence"))\n    if confidence is None:\n        reasons.append("invalid_confidence")\n    elif confidence < POLICY["vietocr_low"]:\n        reasons.append("low_confidence")\n    words = normalized(text).split()\n    if any(words[i:i+n] * 4 == words[i:i+4*n]\n           for n in (1, 2, 3) for i in range(max(0, len(words) - 4*n + 1))):\n        reasons.append("repeated_phrase")\n    if len(text) >= POLICY["decode_limit"]:\n        reasons.append("decode_limit")\n    return reasons\n\n\ndef numeric(text):\n    text = text.strip()\n    if not re.fullmatch(r"[+-]?[0-9]+(?:[.,:/-][0-9]+)*(?:\\s*[%₫$€])?", text):\n        return False\n    if re.fullmatch(r"\\d{1,2}:\\d{2}(?::\\d{2})?", text):\n        parts = list(map(int, text.split(":")))\n        return parts[0] < 24 and all(p < 60 for p in parts[1:])\n    return True\n\n\ndef paddle_candidate(viet, cache_text):\n    texts = (viet["text"], cache_text)\n    return bool(guards(viet) or any(numeric(t) or (t.isascii() and re.search(r"[A-Za-z]{2}", t)) for t in texts))\n\n\ndef select_result(viet, paddle, cache_text):\n    reasons = guards(viet)\n    accepted = viet if not reasons else None\n    engine, decision = ("vietocr", "vietocr_default") if accepted else (None, "unresolved")\n    if paddle is not None and not guards(paddle) and score(paddle["confidence"]) >= POLICY["paddle_override"]:\n        text = paddle["text"]\n        digits = lambda t: "".join(re.findall(r"[0-9]", t))\n        if numeric(text) and any(numeric(t) and digits(t) == digits(text) for t in (viet["text"], cache_text)):\n            accepted, engine, decision, reasons = paddle, "paddle", "numeric_cache_or_viet_guard", []\n        elif reasons and text.isascii() and re.search(r"[A-Za-z]{2}", text) and normalized(text) == normalized(cache_text):\n            accepted, engine, decision, reasons = paddle, "paddle", "ascii_cache_guard", []\n    if paddle is not None and decision == "vietocr_default" and normalized(paddle["text"]) != normalized(viet["text"]):\n        reasons = ["model_disagreement"]\n    return {"selected_text": accepted["text"] if accepted else None,\n            "selected_confidence": accepted["confidence"] if accepted else None,\n            "selected_engine": engine, "selection": decision, "residual_reasons": reasons}\n\n\nclass Journal:\n    """Local FULL-sync SQLite is a worker checkpoint, never the shared FTS database."""\n    def __init__(self, path, signature):\n        self.db = sqlite3.connect(path)\n        self.db.execute("PRAGMA synchronous=FULL")\n        self.db.execute("CREATE TABLE IF NOT EXISTS meta (k TEXT PRIMARY KEY, v TEXT NOT NULL)")\n        self.db.execute("CREATE TABLE IF NOT EXISTS predictions (n INTEGER PRIMARY KEY, model TEXT, region TEXT, payload BLOB, durable INTEGER NOT NULL DEFAULT 0, UNIQUE(model,region))")\n        prior = self.get_meta("signature")\n        if prior is not None and prior != signature:\n            self.db.close()\n            raise ValueError("Checkpoint signature mismatch")\n        self.set_meta("signature", signature)\n\n    def get_meta(self, key):\n        row = self.db.execute("SELECT v FROM meta WHERE k=?", (key,)).fetchone()\n        return row[0] if row else None\n\n    def set_meta(self, key, value):\n        with self.db:\n            self.db.execute("INSERT OR REPLACE INTO meta VALUES (?,?)", (key, str(value)))\n\n    def get(self, model, region):\n        row = self.db.execute("SELECT payload FROM predictions WHERE model=? AND region=?", (model, region)).fetchone()\n        return json.loads(row[0]) if row else None\n\n    def save(self, rows, durable=False):\n        with self.db:\n            for row in rows:\n                key = row["model"], row["region_id"]\n                previous = self.get(*key)\n                if previous is not None and previous != row:\n                    raise ValueError("Conflicting checkpoint predictions")\n                if previous is None:\n                    self.db.execute("INSERT INTO predictions(model,region,payload,durable) VALUES (?,?,?,?)", (*key, encoded(row), int(durable)))\n                elif durable:\n                    self.db.execute("UPDATE predictions SET durable=1 WHERE model=? AND region=?", key)\n\n    def count(self, model=None):\n        if model:\n            return self.db.execute("SELECT count(*) FROM predictions WHERE model=?", (model,)).fetchone()[0]\n        return self.db.execute("SELECT count(*) FROM predictions").fetchone()[0]\n\n    def close(self):\n        self.db.close()\n\n\ndef zip_payload(path, members):\n    buffer = io.BytesIO()\n    with zipfile.ZipFile(buffer, "w", zipfile.ZIP_DEFLATED) as archive:\n        checks = "".join(f"{sha(value)}  {name}\\n" for name, value in sorted(members.items())).encode()\n        for name, payload in sorted({**members, "SHA256SUMS": checks}.items()):\n            info = zipfile.ZipInfo(name, (1980, 1, 1, 0, 0, 0))\n            info.compress_type = zipfile.ZIP_DEFLATED\n            archive.writestr(info, payload)\n    atomic(path, buffer.getvalue())\n\n\nclass DurableJournal:\n    """Immutable delta chunks: avoids reuploading all prior predictions every five minutes."""\n    def __init__(self, journal, hf, prefix, output, signature, expected):\n        self.journal, self.hf, self.prefix, self.output = journal, hf, prefix, Path(output)\n        self.signature, self.expected = signature, expected\n        self.sequence, self.previous, self.last_sync = 0, None, time.monotonic()\n\n    def validate_rows(self, rows):\n        seen = set()\n        for row in rows:\n            key = row.get("model"), row.get("region_id")\n            if key in seen or key[0] not in MODEL_NAMES or key[1] not in self.expected:\n                raise ValueError("Duplicate/foreign checkpoint row")\n            seen.add(key)\n            if (row.get("task_sha256") != self.expected[key[1]] or row.get("signature") != self.signature\n                    or not isinstance(row.get("text"), str) or score(row.get("confidence")) != row.get("confidence")):\n                raise ValueError("Checkpoint task/signature/prediction mismatch")\n\n    def restore(self):\n        for model, region, payload in self.journal.db.execute("SELECT model,region,payload FROM predictions"):\n            row = json.loads(payload)\n            self.validate_rows([row])\n            if (row["model"], row["region_id"]) != (model, region):\n                raise ValueError("Local checkpoint index/payload mismatch")\n        revision = self.hf.revision()\n        chunks = []\n        for name in self.hf.files(revision):\n            if not name.startswith(self.prefix + "/checkpoints/"):\n                continue\n            match = re.fullmatch(r"part-(\\d{6})-([0-9a-f]{64})\\.zip", PurePosixPath(name).name)\n            if not match:\n                raise ValueError("Unexpected checkpoint filename")\n            chunks.append((int(match[1]), name, match[2]))\n        restored_keys = set()\n        for sequence, name, digest in sorted(chunks):\n            if sequence != self.sequence:\n                raise ValueError("HF checkpoint chain gap/conflict; do not duplicate a worker")\n            path = self.hf.download(name, revision)\n            if file_sha(path) != digest:\n                raise ValueError("HF checkpoint checksum mismatch")\n            with zipfile.ZipFile(path) as archive:\n                manifest = json.loads(archive.read("chunk.json"))\n                payload = archive.read("predictions.jsonl")\n            if (manifest["signature"] != self.signature or manifest["previous_sha256"] != self.previous\n                    or manifest["sequence"] != sequence or manifest["rows_sha256"] != sha(payload)):\n                raise ValueError("Invalid checkpoint chain/signature")\n            rows = [json.loads(line) for line in payload.splitlines()]\n            self.validate_rows(rows)\n            keys = {(r["model"], r["region_id"]) for r in rows}\n            if keys & restored_keys or len(rows) != manifest["count"]:\n                raise ValueError("Duplicate remote prediction or count mismatch")\n            restored_keys.update(keys)\n            self.journal.save(rows, durable=True)\n            self.sequence += 1\n            self.previous = digest\n        # Reject a local DB which claims durability absent from the remote chain.\n        local_durable = set(self.journal.db.execute("SELECT model,region FROM predictions WHERE durable=1"))\n        if local_durable != restored_keys:\n            raise ValueError("Local/HF durable sets diverge")\n        self.journal.set_meta("last_hf", self.previous or "none")\n        log("RESUME_VERIFIED", predictions=len(restored_keys), chunks=self.sequence, signature=self.signature)\n        return len(restored_keys)\n\n    def sync(self, force=False):\n        if not force and time.monotonic() - self.last_sync < 300:\n            return\n        if time.monotonic() - self.last_sync >= 300:\n            log("CHECKPOINT_DUE", seconds_since_last_sync=round(time.monotonic() - self.last_sync))\n        rows = [json.loads(r[0]) for r in self.journal.db.execute("SELECT payload FROM predictions WHERE durable=0 ORDER BY n")]\n        if rows:\n            self.validate_rows(rows)\n            payload = b"".join(encoded(row) + b"\\n" for row in rows)\n            manifest = {"signature": self.signature, "previous_sha256": self.previous,\n                        "sequence": self.sequence, "count": len(rows), "rows_sha256": sha(payload)}\n            path = self.output / "pending-checkpoint.zip"\n            zip_payload(path, {"chunk.json": encoded(manifest), "predictions.jsonl": payload})\n            digest = file_sha(path)\n            name = f"{self.prefix}/checkpoints/part-{self.sequence:06d}-{digest}.zip"\n            with heartbeat("HF_SYNC", count=len(rows)):\n                self.hf.put(path, name)\n            self.journal.save(rows, durable=True)\n            self.sequence += 1\n            self.previous = digest\n            self.journal.set_meta("last_hf", digest)\n        self.last_sync = time.monotonic()\n\n\ndef video_directories(root, videos):\n    found = {}\n    for current, dirs, _ in os.walk(root):\n        for name in list(dirs):\n            if re.fullmatch(r"L\\d+_V\\d+", name):\n                dirs.remove(name)\n                if name in videos:\n                    if name in found:\n                        raise ValueError(f"Multiple JPEG directories for {name}")\n                    found[name] = Path(current) / name\n    if set(found) != set(videos):\n        raise FileNotFoundError(f"Attach keyframe datasets for missing videos: {sorted(set(videos) - set(found))[:12]}")\n    return found\n\n\ndef build_tasks(frames, root, mode):\n    from PIL import Image, ImageOps\n    regions = [(f, r) for f in frames for r in f.get("regions", [])]\n    if mode == "canary":\n        regions = sorted(regions, key=lambda pair: sha(pair[1]["region_id"].encode()))[:256]\n    directories = video_directories(root, {f["video_id"] for f, _ in regions})\n    images, tasks = {}, []\n    for frame, region in regions:\n        image_key = frame["video_id"] + "/" + PurePosixPath(frame["source_image"]).name\n        if image_key not in images:\n            path = directories[frame["video_id"]] / PurePosixPath(frame["source_image"]).name\n            with Image.open(path) as opened:\n                size = ImageOps.exif_transpose(opened).size\n            if list(size) != [frame["image_width"], frame["image_height"]]:\n                raise ValueError(f"Source image geometry drift: {image_key}")\n            images[image_key] = {"path": str(path), "sha256": file_sha(path), "size": list(size)}\n            if len(images) % 500 == 0:\n                log("IMAGE_PREFLIGHT", images=len(images), regions=len(tasks))\n        task = {"region_id": region["region_id"], "keyframe_uid": frame["keyframe_uid"],\n                "video_id": frame["video_id"], "frame_id": frame["frame_id"],\n                "shot_id": frame["shot_id"], "source_image": frame["source_image"],\n                "image_key": image_key, "source_sha256": images[image_key]["sha256"],\n                "bbox_px": region["bbox_px"], "easyocr_text": region.get("easyocr_text") or ""}\n        task["task_sha256"] = sha(encoded(task))\n        tasks.append(task)\n    return tasks, images\n\n\ndef environment():\n    names = ("torch", "torchvision", "vietocr", "einops", "paddlepaddle-gpu", "paddleocr", "paddlex", "Pillow", "numpy")\n    versions = {}\n    for name in names:\n        try:\n            versions[name] = md.version(name)\n        except md.PackageNotFoundError:\n            raise RuntimeError(f"Missing {name}; run setup cell first") from None\n    for name, wanted in {"vietocr": "0.3.13", "paddlepaddle-gpu": "3.2.2", "paddleocr": "3.7.0"}.items():\n        if versions[name] != wanted:\n            raise RuntimeError(f"Use pinned {name}=={wanted}; found {versions[name]}")\n    for dist in md.distributions():\n        name = (dist.metadata.get("Name") or "").lower().replace("_", "-")\n        if name.startswith("nvidia-"):\n            versions[name] = dist.version\n    return versions\n\n\ndef model_predictor(model, model_dir):\n    h, _ = helpers(model_dir)\n    if model == "vietocr":\n        import torch\n        from vietocr.tool.predictor import Predictor\n        if not torch.cuda.is_available() or "T4" not in torch.cuda.get_device_name(0).upper():\n            raise RuntimeError("Requires Kaggle T4, no CPU fallback")\n        torch.cuda.reset_peak_memory_stats(0)\n        _, config = h.prepare_vietocr_model()\n        # Guard must agree with the real pinned VietOCR decoder limit.\n        if int(config.get("predictor", {}).get("max_seq_length", 128)) != POLICY["decode_limit"]:\n            raise ValueError("VietOCR decode limit drift")\n        predictor = Predictor(config)\n        def predict(images):\n            with torch.inference_mode():\n                texts, probabilities = predictor.predict_batch(images, return_prob=True)\n            if len(texts) != len(images) or len(probabilities) != len(images):\n                raise ValueError("Incomplete VietOCR minibatch")\n            return [(str(text), score(prob)) for text, prob in zip(texts, probabilities)]\n        predict.hardware = lambda: {"gpu": torch.cuda.get_device_name(0), "cuda": torch.version.cuda,\n                                    "peak_vram_mb": torch.cuda.max_memory_allocated(0) / (1024 * 1024)}\n        return predict, torch.cuda.empty_cache\n    import paddle\n    import numpy as np\n    from paddleocr import TextRecognition\n    if not paddle.is_compiled_with_cuda() or "T4" not in paddle.device.cuda.get_device_name(0).upper():\n        raise RuntimeError("Requires Paddle CUDA on Kaggle T4, no CPU fallback")\n    paddle.device.set_device("gpu:0")\n    model_dir = h.prepare_paddle_model()\n    predictor = TextRecognition(model_name=h.PADDLE_MODEL_ID, model_dir=str(model_dir), device="gpu:0")\n    def predict(images):\n        arrays = [np.asarray(im, dtype=np.uint8)[:, :, ::-1] for im in images]\n        results = list(predictor.predict(input=arrays, batch_size=len(arrays)))\n        if len(results) != len(images):\n            raise ValueError("Incomplete Paddle minibatch")\n        parsed = []\n        for row in results:\n            value = row.json\n            value = value() if callable(value) else value\n            value = json.loads(value) if isinstance(value, str) else value\n            value = value.get("res", value)\n            parsed.append((str(value.get("rec_text") or ""), score(value.get("rec_score"))))\n        return parsed\n    def hardware():\n        try:\n            peak = float(paddle.device.cuda.max_memory_allocated()) / (1024 * 1024)\n        except (AttributeError, RuntimeError):\n            peak = None\n        return {"gpu": paddle.device.cuda.get_device_name(0), "peak_vram_mb": peak}\n    predict.hardware = hardware\n    return predict, paddle.device.cuda.empty_cache\n\n\ndef task_crops(tasks, images, rectify):\n    from PIL import Image, ImageOps\n    loaded, crops = {}, []\n    try:\n        for task in tasks:\n            key = task["image_key"]\n            if key not in loaded:\n                payload = Path(images[key]["path"]).read_bytes()\n                if sha(payload) != task["source_sha256"]:\n                    raise ValueError(f"Source image changed after preflight: {key}")\n                with Image.open(io.BytesIO(payload)) as opened:\n                    loaded[key] = ImageOps.exif_transpose(opened).convert("RGB")\n            crops.append(rectify(loaded[key], task["bbox_px"]))\n        return crops\n    finally:\n        for im in loaded.values():\n            im.close()\n\n\nclass IntentionalStop(RuntimeError):\n    pass\n\n\ndef recognize(tasks, model, journal, durable, predict, get_crops, *, signature,\n              batch_size, release=lambda: None, interrupt_after=0, identity=None):\n    identity = identity or {}\n    before = journal.count(model)\n    durable_before = journal.db.execute("SELECT count(*) FROM predictions WHERE model=? AND durable=1", (model,)).fetchone()[0]\n    eligible = []\n    for task in tasks:\n        if model == "paddle":\n            viet = journal.get("vietocr", task["region_id"])\n            if viet is None:\n                raise ValueError("Paddle cannot run before VietOCR coverage is complete")\n            if not paddle_candidate(viet, task["easyocr_text"]):\n                continue\n        eligible.append(task)\n    pending = [t for t in eligible if journal.get(model, t["region_id"]) is None]\n    started, cursor, minibatches = time.monotonic(), 0, 0\n    log("PHASE_START", model=model, total=len(eligible), resumed=before, pending=len(pending), **identity)\n    while cursor < len(pending):\n        durable.sync()\n        batch = pending[cursor:cursor + batch_size]\n        crops = get_crops(batch)\n        try:\n            predictions = predict(crops)\n        except Exception as exc:\n            is_oom = "out of memory" in str(exc).lower() or "resourceexhausted" in type(exc).__name__.lower()\n            if not is_oom or batch_size == 1:\n                durable.sync(force=True)\n                raise\n            release()\n            batch_size = max(1, batch_size // 2)\n            log("OOM_REDUCE_BATCH", model=model, batch_size=batch_size, **identity)\n            continue\n        finally:\n            for crop in crops:\n                crop.close()\n        if len(predictions) != len(batch):\n            raise ValueError("Incomplete predictions; minibatch not saved")\n        rows = [{"model": model, "region_id": t["region_id"], "task_sha256": t["task_sha256"],\n                 "signature": signature, "text": text, "confidence": score(confidence)}\n                for t, (text, confidence) in zip(batch, predictions)]\n        durable.validate_rows(rows)\n        journal.save(rows)\n        cursor += len(batch)\n        minibatches += 1\n        elapsed = time.monotonic() - started\n        rate = cursor / max(elapsed, 1e-9)\n        log("MINIBATCH_SAVED", model=model, video=batch[-1]["video_id"], done=before + cursor,\n            total=len(eligible), new=cursor, elapsed=round(elapsed, 2), regions_per_second=round(rate, 2),\n            eta_seconds=round((len(pending) - cursor) / rate), last_hf=journal.get_meta("last_hf"), **identity)\n        if durable_before and cursor:\n            journal.set_meta("resume_newwork", "true")\n        if interrupt_after and minibatches >= interrupt_after:\n            durable.sync(force=True)\n            raise IntentionalStop("Checkpoint verified on HF. Set INTERRUPT_AFTER_MINIBATCHES=0 and rerun in a new process/session.")\n    durable.sync(force=True)\n    return {"model": model, "expected": len(eligible), "completed": journal.count(model),\n            "new_predictions": cursor, "resumed_predictions": before,\n            "resume_with_new_work": journal.get_meta("resume_newwork") == "true",\n            "recognition_and_sync_seconds": time.monotonic() - started}\n\n\ndef phase(context_path, model):\n    ctx = json.loads(Path(context_path).read_bytes())\n    output = Path(ctx["output"])\n    hf = HF(ctx["repo"])\n    tasks = [json.loads(line) for line in (output / "tasks.jsonl").read_bytes().splitlines()]\n    if sha(b"".join(encoded(t) + b"\\n" for t in tasks)) != ctx["tasks_sha256"]:\n        raise ValueError("Task manifest drift")\n    journal = Journal(output / "checkpoint.sqlite", ctx["signature"])\n    durable = DurableJournal(journal, hf, ctx["prefix"], output, ctx["signature"],\n                             {t["region_id"]: t["task_sha256"] for t in tasks})\n    try:\n        with heartbeat("HF_RESTORE", model=model, worker=ctx["worker"], batch=ctx["batch"]):\n            durable.restore()\n        # Verify write access before expensive model init, including empty/no-candidate phases.\n        preflight = output / "run-signature.json"\n        hf.put(preflight, f"{ctx[\'prefix\']}/preflight-{file_sha(preflight)}.json")\n        initialized = []\n        def lazy_predict(crops):\n            if not initialized:\n                with heartbeat("MODEL_INIT", model=model):\n                    initialized.extend(model_predictor(model, output.parent.parent / "models"))\n            return initialized[0](crops)\n        h, _ = helpers(output.parent.parent / "models")\n        with heartbeat("RECOGNITION", model=model, worker=ctx["worker"], batch=ctx["batch"]):\n            report = recognize(tasks, model, journal, durable, lazy_predict,\n                lambda group: task_crops(group, ctx["images"], h.rectify_region_crop),\n                signature=ctx["signature"], batch_size=POLICY[model + "_batch"],\n                release=lambda: initialized[1]() if initialized else None,\n                interrupt_after=ctx["interrupt_after"], identity={"worker": ctx["worker"], "batch": ctx["batch"]})\n        report["hardware"] = initialized[0].hardware() if initialized and hasattr(initialized[0], "hardware") else None\n        atomic(output / f"{model}-phase.json", encoded(report))\n    finally:\n        journal.close()\n\n\ndef export_results(ctx, frames, tasks, hf, started):\n    output = Path(ctx["output"])\n    journal = Journal(output / "checkpoint.sqlite", ctx["signature"])\n    try:\n        selections = {}\n        predictions = [json.loads(row[0]) for row in journal.db.execute("SELECT payload FROM predictions ORDER BY model,region")]\n        for task in tasks:\n            rid = task["region_id"]\n            viet, paddle = journal.get("vietocr", rid), journal.get("paddle", rid)\n            if viet is None or (paddle_candidate(viet, task["easyocr_text"]) and paddle is None):\n                raise ValueError("Cannot export incomplete recognizer coverage")\n            if not paddle_candidate(viet, task["easyocr_text"]) and paddle is not None:\n                raise ValueError("Unexpected Paddle prediction")\n            selections[rid] = {**task, **select_result(viet, paddle, task["easyocr_text"])}\n        results, residuals = [], []\n        for frame in frames:\n            source_regions = frame.get("regions") or []\n            if ctx["mode"] == "canary" and not any(r["region_id"] in selections for r in source_regions):\n                continue\n            regions = [selections[r["region_id"]] for r in source_regions if r["region_id"] in selections]\n            accepted = [r for r in regions if r["selected_text"] is not None]\n            status = "success" if accepted else ("no_text" if frame["status"] == "no_text" else "error")\n            result = None\n            if accepted:\n                boxes = []\n                for r in accepted:\n                    boxes.append([max(0.0, min(1.0, float(v) / (frame["image_width"] if i % 2 == 0 else frame["image_height"])))\n                                  for i, v in enumerate(r["bbox_px"])])\n                weights = [max(1, len("".join(r["selected_text"].split()))) for r in accepted]\n                confidence = sum(w * r["selected_confidence"] for w, r in zip(weights, accepted)) / sum(weights)\n                result = {"frame_id": frame["frame_id"], "detected_text": [r["selected_text"] for r in accepted],\n                          "bbox": boxes, "confidence": confidence, "language": "mixed"}\n            residuals.extend(r for r in regions if r["residual_reasons"])\n            results.append({"artifact_kind": "ocr_v2_frame_selection_v1", "batch_id": ctx["batch"],\n                "signature": ctx["signature"], "keyframe_uid": frame["keyframe_uid"], "video_id": frame["video_id"],\n                "frame_id": frame["frame_id"], "source_image": frame["source_image"], "status": status,\n                "result": result, "regions": regions, "source_status": frame["status"],\n                "source_error": frame.get("error"), "complete": False, "production_ready": False})\n        report = {"contract": CONTRACT, "run_id": ctx["run_id"], "signature": ctx["signature"],\n                  "mode": ctx["mode"], "worker": ctx["worker"], "batch": ctx["batch"],\n                  "tasks_sha256": ctx["tasks_sha256"], "sample_task_sha256": ctx["sample_task_sha256"],\n                  "regions": len(tasks), "predictions": len(predictions), "frames": len(results),\n                  "status": dict(Counter(r["status"] for r in results)), "residual_regions": len(residuals),\n                  "residual_frames": len({r["keyframe_uid"] for r in residuals}),\n                  "residual_shots": len({(r["video_id"], r["shot_id"]) for r in residuals}),\n                  "recognition_complete": True, "complete": False, "production_ready": False,\n                  "resume_with_new_work": journal.get_meta("resume_newwork") == "true",\n                  "end_to_end_seconds_this_run": time.monotonic() - started,\n                  "phases": {model: json.loads((output / f"{model}-phase.json").read_bytes())\n                             if (output / f"{model}-phase.json").is_file() else None for model in MODEL_NAMES},\n                  "model_calls_saved": dict(Counter(r["model"] for r in predictions)),\n                  "other_model_calls": {"easyocr": 0, "vintern": 0, "gemini": 0},\n                  "limitations": ["Not a quantitative accuracy gate; canary is not a representative sample.",\n                                  "Not a legacy OcrRecordEnvelope or Online snapshot; migration/union still required.",\n                                  "Resume timing excludes prior process runs; do not extrapolate a resumed run as full throughput.",\n                                  "Language is mixed/undetermined; no ASCII-as-English classifier."]}\n        members = {"report.json": encoded(report), "run-signature.json": (output / "run-signature.json").read_bytes()}\n        for name, rows in (("predictions.jsonl", predictions), ("frame-selections.jsonl", results), ("residual.jsonl", residuals)):\n            members[name] = b"".join(encoded(row) + b"\\n" for row in rows)\n        archive = output / f"ocr-v2-{ctx[\'batch\']}-{ctx[\'mode\']}-results.zip"\n        zip_payload(archive, members)\n        report_path = output / "report.json"\n        atomic(report_path, members["report.json"])\n        with heartbeat("RESULT_UPLOAD", worker=ctx["worker"], batch=ctx["batch"]):\n            hf.put(archive, f"{ctx[\'prefix\']}/results-{file_sha(archive)}.zip")\n            hf.put(report_path, f"{ctx[\'prefix\']}/reports/summary-{file_sha(report_path)}.json")\n        log("WORKER_BATCH_COMPLETE", output=str(archive), report_sha256=file_sha(report_path), **report)\n        return report\n    finally:\n        journal.close()\n\n\ndef run_worker(config, hf):\n    plan = json.loads(Path(config["plan"]).read_bytes())\n    validate_plan(plan)\n    if hf.repo != plan["repo"]:\n        raise ValueError("HF repo differs from worker plan")\n    worker = str(config["worker"])\n    if worker not in plan["assignments"] or config["mode"] not in {"canary", "production"}:\n        raise ValueError("Choose worker 1–4 and canary/production mode")\n    catalog_path = resolve_catalog(config)\n    if any(plan[key] != value for key, value in catalog_hashes(catalog_path).items()):\n        raise ValueError("Worker catalog hash drift")\n    catalog = load_catalog(catalog_path)\n    versions = environment()\n    h, reference_hash = helpers(Path(config["output"]) / "models")\n    resources = {"contract": CONTRACT, "runtime_sha256": file_sha(__file__),\n                 "reference_sha256": reference_hash, "policy": POLICY, "packages": versions,\n                 "plan_sha256": plan["plan_sha256"], "worker": worker,\n                 "vietocr_weight": h.VIETOCR_WEIGHT, "vietocr_configs": h.VIETOCR_CONFIGS, "paddle": h.PADDLE_MODEL}\n    run_id = sha(encoded(resources))\n    batches = plan["assignments"][worker]\n    if config["mode"] == "canary":\n        batches = batches[:1]\n    for batch in batches:\n        started = time.monotonic()\n        with heartbeat("INPUT_VALIDATE", worker=worker, batch=batch):\n            evidence = plan["batches"][batch]\n            archive = hf.download(evidence["archive"], plan["input_revision"])\n            if file_sha(archive) != evidence["sha256"]:\n                raise ValueError("Input archive changed")\n            manifest, frames = load_archive(archive, catalog)\n            if manifest["catalog_sha256"] != plan["catalog_sha256"] or manifest["batch_id"] != batch:\n                raise ValueError("Worker input identity drift")\n            if (manifest["regions"] != evidence["regions"] or manifest["frames"] != evidence["frames"]\n                    or manifest["observed_uid_sha256"] != evidence["uid_sha256"]\n                    or sorted({f["video_id"] for f in frames}) != evidence["video_ids"]):\n                raise ValueError("Worker archive/plan coverage drift")\n        with heartbeat("IMAGE_PREFLIGHT", worker=worker, batch=batch):\n            tasks, images = build_tasks(frames, config["keyframes"], config["mode"])\n        sample = sorted(tasks, key=lambda t: sha(t["region_id"].encode()))[:256]\n        sample_sha = sha(encoded([t["task_sha256"] for t in sample]))\n        if config["mode"] == "canary" and len(tasks) < 128:\n            raise ValueError("Canary needs at least 128 regions for interrupt/new-work proof")\n        if config["mode"] == "production":\n            approved = config.get("approved_canary_sha256", "")\n            if not re.fullmatch(r"[0-9a-f]{64}", approved):\n                raise ValueError("Run canary first, then copy report_sha256 into APPROVED_CANARY_SHA256")\n            # Canary on the first assigned batch authorizes only this worker/config/runtime.\n            name = f"ocr/archives/{batches[0]}/ocr-v2/{run_id}/canary/reports/summary-{approved}.json"\n            report_path = hf.download(name, hf.revision())\n            report = json.loads(report_path.read_bytes())\n            if (file_sha(report_path) != approved or report.get("run_id") != run_id\n                    or report.get("mode") != "canary" or report.get("recognition_complete") is not True\n                    or report.get("resume_with_new_work") is not True\n                    or (batch == batches[0] and report.get("sample_task_sha256") != sample_sha)):\n                raise ValueError("Canary is incomplete, has no resume/new-work proof, or source/config changed")\n        task_bytes = b"".join(encoded(t) + b"\\n" for t in tasks)\n        signature = sha(encoded({"run_id": run_id, "batch": batch, "mode": config["mode"], "tasks": sha(task_bytes)}))\n        output = Path(config["output"]) / run_id / batch / config["mode"]\n        output.mkdir(parents=True, exist_ok=True)\n        prefix = f"ocr/archives/{batch}/ocr-v2/{run_id}/{config[\'mode\']}"\n        ctx = {"run_id": run_id, "signature": signature, "batch": batch, "worker": worker,\n               "mode": config["mode"], "repo": hf.repo, "output": str(output), "prefix": prefix,\n               "images": images, "tasks_sha256": sha(task_bytes), "sample_task_sha256": sample_sha,\n               "interrupt_after": int(config.get("interrupt_after", 0))}\n        atomic(output / "tasks.jsonl", task_bytes)\n        atomic(output / "run-signature.json", encoded({"resources": resources, "signature": signature,\n                                                        "tasks_sha256": sha(task_bytes), "batch": batch, "mode": config["mode"]}))\n        atomic(output / "context.json", encoded(ctx))\n        env = {**os.environ, "HF_TOKEN": hf.token, "CUDA_VISIBLE_DEVICES": "0", "TOKENIZERS_PARALLELISM": "false"}\n        for model in MODEL_NAMES:\n            log("LAUNCH_PHASE", worker=worker, batch=batch, model=model)\n            process = subprocess.run([sys.executable, str(Path(__file__).resolve()), "phase",\n                                      str(output / "context.json"), "--model", model], env=env)\n            if process.returncode == 75:\n                log("INTENTIONAL_STOP", worker=worker, batch=batch,\n                    action="Set INTERRUPT_AFTER_MINIBATCHES=0, rerun; checkpoint already verified on HF.")\n                return\n            if process.returncode:\n                raise RuntimeError(f"{model} phase failed; fix error then rerun same signature")\n        export_results(ctx, frames, tasks, hf, started)\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("action", choices=("plan", "run", "phase"))\n    parser.add_argument("config", type=Path)\n    parser.add_argument("--model", choices=MODEL_NAMES)\n    args = parser.parse_args()\n    if args.action == "phase":\n        if not args.model:\n            parser.error("phase requires --model")\n        try:\n            phase(args.config, args.model)\n        except IntentionalStop as exc:\n            log("INTENTIONAL_STOP", action=str(exc))\n            raise SystemExit(75)\n        return\n    config = json.loads(args.config.read_bytes())\n    hf = HF(config["repo"])\n    with heartbeat(args.action, worker=config.get("worker")):\n        if args.action == "plan":\n            create_plan(config, hf)\n        else:\n            run_worker(config, hf)\n\n\nif __name__ == "__main__":\n    main()\n', 'kaggle_ocr_v2_gate_b_runtime.py': "PADDLE_MODEL_ID = 'latin_PP-OCRv5_mobile_rec'\nCROP_SPEC_ID = 'pil_quad_v1_pad08_edge'\nPADDLE_MODEL = {'url': 'https://paddle-model-ecology.bj.bcebos.com/paddlex/official_inference_model/paddle3.0.0/latin_PP-OCRv5_mobile_rec_infer.tar', 'sha256': 'b23105a6a1ea38e32a97c5a0ddc7e8a9bbf541d8e47421e2c99e9ccabe29509c', 'bytes': 8202240}\nVIETOCR_WEIGHT = {'url': 'https://github.com/pbcquoc/vietocr/releases/download/v0.3.2/vgg-seq2seq.pth', 'sha256': '0921503a41375a0584268e23ef3d414ea478a8fe8777865c7745d38f2d0bc5db', 'bytes': 89575371}\nVIETOCR_CONFIGS = {'base.yml': {'url': 'https://raw.githubusercontent.com/pbcquoc/vietocr/fe8c3a7fc714aec57ab81cec844eb3adf0c1636c/config/base.yml', 'sha256': '9c8283fadb950f06f5d3400475f80d5355700ff315c9c48b7875e6ea66647d1c'}, 'vgg-seq2seq.yml': {'url': 'https://raw.githubusercontent.com/pbcquoc/vietocr/fe8c3a7fc714aec57ab81cec844eb3adf0c1636c/config/vgg-seq2seq.yml', 'sha256': '0160ba8d442ae96f4c6095b92ac3521c59b83ce6eda9fd5459e8628a5586c3e8'}}\n\ndef sha256_file(path: Path) -> str:\n    digest = hashlib.sha256()\n    with Path(path).open('rb') as handle:\n        for block in iter(lambda: handle.read(8 * 1024 * 1024), b''):\n            digest.update(block)\n    return digest.hexdigest()\n\ndef _float_list(value: Any) -> list[float]:\n    if isinstance(value, str):\n        value = json.loads(value)\n    if not isinstance(value, list) or len(value) != 8:\n        raise RuntimeError('bbox_px must contain eight values')\n    values = [float(item) for item in value]\n    if not all((math.isfinite(item) for item in values)):\n        raise RuntimeError('bbox_px contains NaN/Inf')\n    return values\n\ndef _edge_pad(image: Image.Image, border: int) -> Image.Image:\n    if border <= 0:\n        return image\n    width, height = image.size\n    padded = Image.new(image.mode, (width + 2 * border, height + 2 * border))\n    padded.paste(image, (border, border))\n    padded.paste(image.crop((0, 0, width, 1)).resize((width, border)), (border, 0))\n    padded.paste(image.crop((0, height - 1, width, height)).resize((width, border)), (border, border + height))\n    padded.paste(image.crop((0, 0, 1, height)).resize((border, height)), (0, border))\n    padded.paste(image.crop((width - 1, 0, width, height)).resize((border, height)), (border + width, border))\n    padded.paste(Image.new(image.mode, (border, border), image.getpixel((0, 0))), (0, 0))\n    padded.paste(Image.new(image.mode, (border, border), image.getpixel((width - 1, 0))), (border + width, 0))\n    padded.paste(Image.new(image.mode, (border, border), image.getpixel((0, height - 1))), (0, border + height))\n    padded.paste(Image.new(image.mode, (border, border), image.getpixel((width - 1, height - 1))), (border + width, border + height))\n    return padded\n\ndef rectify_region_crop(image: Image.Image, bbox_px: list[float]) -> Image.Image:\n    values = _float_list(bbox_px)\n    points = [(values[index], values[index + 1]) for index in range(0, 8, 2)]\n\n    def distance(left: tuple[float, float], right: tuple[float, float]) -> float:\n        return math.hypot(left[0] - right[0], left[1] - right[1])\n    width = max(2, round(max(distance(points[0], points[1]), distance(points[3], points[2]))))\n    height = max(2, round(max(distance(points[0], points[3]), distance(points[1], points[2]))))\n    quad = (points[0][0], points[0][1], points[3][0], points[3][1], points[2][0], points[2][1], points[1][0], points[1][1])\n    crop = image.convert('RGB').transform((width, height), Image.Transform.QUAD, quad, resample=Image.Resampling.BICUBIC)\n    return _edge_pad(crop, max(1, round(height * 0.08)))\n\ndef download_verified(spec: dict[str, Any], destination: Path) -> Path:\n    if not destination.exists() or sha256_file(destination) != spec['sha256']:\n        print('MODEL_DOWNLOAD_START', {'file': destination.name, 'expected_bytes': int(spec['bytes'])}, flush=True)\n        urllib.request.urlretrieve(spec['url'], destination)\n    if destination.stat().st_size != int(spec['bytes']) or sha256_file(destination) != spec['sha256']:\n        raise RuntimeError(f'download checksum/size mismatch: {destination.name}')\n    print('MODEL_ARTIFACT_READY', {'file': destination.name, 'bytes': destination.stat().st_size}, flush=True)\n    return destination\n\ndef prepare_paddle_model() -> Path:\n    archive_path = download_verified(PADDLE_MODEL, OUTPUT_ROOT / 'paddle-model.tar')\n    extract_root = OUTPUT_ROOT / 'paddle-model'\n    extract_root.mkdir(exist_ok=True)\n    with tarfile.open(archive_path) as archive:\n        members = archive.getmembers()\n        if not members or any((member.islnk() or member.issym() or Path(member.name).is_absolute() or ('..' in Path(member.name).parts) for member in members)):\n            raise RuntimeError('unsafe Paddle model archive')\n        archive.extractall(extract_root, members=members, filter='data')\n    model_dirs = sorted({path.parent for path in extract_root.rglob('inference.json')})\n    if len(model_dirs) != 1 or not (model_dirs[0] / 'inference.pdiparams').is_file():\n        raise RuntimeError('unexpected Paddle inference archive layout')\n    return model_dirs[0]\n\ndef prepare_vietocr_model() -> tuple[Path, dict[str, Any]]:\n    import yaml\n    weight_path = download_verified(VIETOCR_WEIGHT, OUTPUT_ROOT / 'vgg-seq2seq.pth')\n    config_values: dict[str, dict[str, Any]] = {}\n    for name, spec in VIETOCR_CONFIGS.items():\n        path = OUTPUT_ROOT / name\n        if not path.exists() or sha256_file(path) != spec['sha256']:\n            urllib.request.urlretrieve(spec['url'], path)\n        if sha256_file(path) != spec['sha256']:\n            raise RuntimeError(f'VietOCR config checksum mismatch: {name}')\n        config_values[name] = yaml.safe_load(path.read_text(encoding='utf-8'))\n    config = dict(config_values['base.yml'])\n    config.update(config_values['vgg-seq2seq.yml'])\n    config['device'] = 'cuda:0'\n    config['weights'] = str(weight_path)\n    config['predictor'] = {'beamsearch': False}\n    config['cnn'] = {**config['cnn'], 'pretrained': False}\n    return (weight_path, config)\n", 'ocr_v2_environment.py': '"""Kaggle-only setup, preserving the preloaded Torch/NVIDIA/NumPy/Pillow stack.\n\nRun by the production notebook, never implicitly on import or in local CPU tests.\n"""\nimport importlib.metadata as md\nimport json\nimport os\nimport subprocess\nimport sys\nfrom pathlib import Path\n\nfrom kaggle_ocr_v2_production_runtime import atomic, file_sha, heartbeat, log\n\nVIETOCR_WHEEL_SHA256 = "07b3777e5176b0d733cb056b68bd817371605f4b3514795fbf91ad4e181b8ccf"\n\n\ndef protected():\n    versions = {}\n    for dist in md.distributions():\n        name = (dist.metadata.get("Name") or "").lower().replace("_", "-")\n        if name in {"torch", "torchvision", "torchaudio", "numpy", "pillow"} or name.startswith("nvidia-"):\n            versions[name] = dist.version\n    return versions\n\n\ndef setup(root):\n    root = Path(root)\n    root.mkdir(parents=True, exist_ok=True)\n    os.environ["CUDA_VISIBLE_DEVICES"] = "0"\n    before = protected()\n    if not {"torch", "torchvision", "numpy", "pillow"} <= set(before):\n        raise RuntimeError("Use a fresh Kaggle GPU image with Torch/torchvision/NumPy/Pillow")\n    pip = [sys.executable, "-m", "pip", "--disable-pip-version-check"]\n    def command(args):\n        with heartbeat("ENV_INSTALL", package=args[-1]):\n            subprocess.run(pip + args, check=True, timeout=1200)\n        if protected() != before:\n            raise RuntimeError("Protected GPU/image stack changed; stop and use a fresh session")\n    log("ENV_1_GPU_RUNTIME", protected=before)\n    command(["install", "--no-deps", "--timeout", "120", "--retries", "3",\n             "--index-url", "https://www.paddlepaddle.org.cn/packages/stable/cu126/", "paddlepaddle-gpu==3.2.2"])\n    log("ENV_2_VIETOCR_WHEEL")\n    command(["download", "--no-deps", "--only-binary=:all:", "--dest", str(root), "vietocr==0.3.13"])\n    wheels = list(root.glob("vietocr-0.3.13-*.whl"))\n    if len(wheels) != 1 or file_sha(wheels[0]) != VIETOCR_WHEEL_SHA256:\n        raise ValueError("VietOCR wheel checksum mismatch")\n    command(["install", "--no-deps", str(wheels[0])])\n    # Resolve the Python dependencies (PaddleX included) without changing GPU/image wheels.\n    constraints = root / "protected-constraints.txt"\n    pins = {**before, "paddlepaddle-gpu": "3.2.2", "vietocr": "0.3.13"}\n    atomic(constraints, "".join(f"{name}=={version}\\n" for name, version in sorted(pins.items())).encode())\n    requests = ["paddleocr==3.7.0", "einops==0.8.1", "gdown==5.2.0", "PyYAML==6.0.2", "huggingface_hub"]\n    report = root / "dependency-resolution.json"\n    log("ENV_3_RESOLVE_PYTHON_DEPENDENCIES")\n    command(["install", "--dry-run", "--report", str(report), "--constraint", str(constraints)] + requests)\n    install = json.loads(report.read_bytes())["install"]\n    resolved = []\n    for row in install:\n        metadata = row["metadata"]\n        name = metadata["name"].lower().replace("_", "-")\n        if name in pins or name.startswith("nvidia-") or name in {"torch", "torchvision", "torchaudio", "paddlepaddle"}:\n            raise RuntimeError(f"Resolver wants to replace/add protected runtime {name}; no installation performed")\n        resolved.append(f"{metadata[\'name\']}=={metadata[\'version\']}")\n    if resolved:\n        command(["install", "--constraint", str(constraints)] + resolved)\n    log("ENV_4_SEPARATE_GPU_PROBES")\n    probes = [\n        "import torch; from vietocr.tool.predictor import Predictor; assert torch.cuda.is_available(); "\n        "assert \'T4\' in torch.cuda.get_device_name(0).upper(); print(\'VIETOCR_T4_IMPORT_OK\')",\n        "import paddle; from paddleocr import TextRecognition; assert paddle.is_compiled_with_cuda(); "\n        "paddle.device.set_device(\'gpu:0\'); assert \'T4\' in paddle.device.cuda.get_device_name(0).upper(); "\n        "print(\'PADDLE_T4_IMPORT_OK\')",\n    ]\n    for probe in probes:\n        with heartbeat("GPU_IMPORT_PROBE"):\n            result = subprocess.run([sys.executable, "-c", probe], text=True, capture_output=True, timeout=180)\n        if result.returncode:\n            raise RuntimeError("GPU probe failed; inference not started:\\n" + result.stderr)\n        print(result.stdout, flush=True)\n    log("ENV_READY", protected_unchanged=protected() == before)\n\n\nif __name__ == "__main__":\n    setup(sys.argv[1])\n'}


In [ ]:
import json, os, signal, subprocess, sys, threading, time
from contextlib import contextmanager
from pathlib import Path
from kaggle_secrets import UserSecretsClient

@contextmanager
def progress(stage, interval=20):
    started = time.monotonic()
    stop = threading.Event()
    print(f'[START] {stage}', flush=True)
    def tick():
        while not stop.wait(interval):
            print(f'[WAIT] {stage} | elapsed={time.monotonic() - started:.0f}s | '
                  'still waiting; progress/ETA unavailable unless reported below', flush=True)
    thread = threading.Thread(target=tick, daemon=True)
    thread.start()
    try:
        yield
    except BaseException as exc:
        print(f'[STOP] {stage} | {type(exc).__name__} | elapsed={time.monotonic() - started:.0f}s', flush=True)
        raise
    else:
        print(f'[DONE] {stage} | elapsed={time.monotonic() - started:.0f}s', flush=True)
    finally:
        stop.set()
        thread.join()

def run_logged(command, stage, env):
    # Forward child stdout/stderr into notebook output; inherited OS stdout may be invisible.
    with progress(stage):
        process = subprocess.Popen(command, env=env, stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT, text=True, encoding='utf-8', errors='replace',
            bufsize=1, start_new_session=(os.name != 'nt'))
        try:
            for line in process.stdout:
                secret = env.get('HF_TOKEN')
                print(line.replace(secret, '[REDACTED]') if secret else line, end='', flush=True)
            returncode = process.wait()
            if returncode:
                raise subprocess.CalledProcessError(returncode, command)
        except BaseException:
            # Stop the Linux process group too, including pip/model subprocesses.
            if os.name != 'nt':
                try:
                    os.killpg(process.pid, signal.SIGTERM)
                except ProcessLookupError:
                    pass
            elif process.poll() is None:
                process.terminate()
            try:
                process.wait(timeout=5)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
            if os.name != 'nt':
                try:
                    os.killpg(process.pid, signal.SIGKILL)
                except ProcessLookupError:
                    pass
            raise
        finally:
            process.stdout.close()

root = Path(os.environ.get('AIC_DATA', '/kaggle/working')) / 'ocr-v2-production'
code_dir = root / 'runtime'
with progress('1/5 Write runtime files'):
    code_dir.mkdir(parents=True, exist_ok=True)
    for name, source in SOURCES.items():
        (code_dir / name).write_text(source, encoding='utf-8')
if ACTION not in ('plan', 'run'):
    raise ValueError('ACTION must be plan/run')
with progress('2/5 Locate worker plan'):
    plan_path = WORKER_PLAN
    if ACTION == 'run' and not plan_path:
        matches = list(Path(INPUT_ROOT).rglob('ocr-v2-worker-plan.json'))
        if len(matches) != 1:
            raise ValueError('Attach exactly one worker plan or set WORKER_PLAN')
        plan_path = str(matches[0])
    if ACTION == 'run':
        if not Path(plan_path).is_file():
            raise ValueError('WORKER_PLAN does not exist: ' + plan_path)
        print('WORKER_PLAN', plan_path, flush=True)
with progress('3/5 Read Kaggle Secret HF_TOKEN'):
    token = UserSecretsClient().get_secret('HF_TOKEN')
child_env = {**os.environ, 'HF_TOKEN': token, 'CUDA_VISIBLE_DEVICES': '0',
             'TOKENIZERS_PARALLELISM': 'false', 'PYTHONUNBUFFERED': '1'}
if ACTION == 'run' and RUN_SETUP:
    run_logged([sys.executable, '-u', str(code_dir / 'ocr_v2_environment.py'), str(root / 'env-cache')],
               '4/5 Environment setup and GPU probes', child_env)
else:
    print('[SKIP] 4/5 Environment setup', flush=True)
config = {'repo': HF_REPO_ID, 'input_revision': INPUT_REVISION,
          'catalog_path': CATALOG_PATH, 'input_root': INPUT_ROOT,
          'output': str(root), 'plan': plan_path, 'keyframes': KEYFRAMES_ROOT,
          'worker': WORKER_SLOT, 'mode': RUN_MODE, 'interrupt_after': INTERRUPT_AFTER_MINIBATCHES,
          'approved_canary_sha256': APPROVED_CANARY_SHA256}
config_path = root / 'worker-config.json'
config_path.write_text(json.dumps(config), encoding='utf-8')
run_logged([sys.executable, '-u', str(code_dir / 'kaggle_ocr_v2_production_runtime.py'), ACTION, str(config_path)],
           f'5/5 OCR {ACTION} | worker={WORKER_SLOT} | mode={RUN_MODE}', child_env)
print('OUTPUT_ROOT', root, flush=True)
